# Breast Cancer Classification

Refined notebook for the classification task using the Breast Cancer Wisconsin dataset. It presents 
data loading, preprocessing, EDA, training, evaluation metrics and visualizations.

## 1. Imports and helper utilities
This section imports all necessary libraries for the analysis. We use scikit-learn for machine learning models and preprocessing, pandas and numpy for data manipulation, and matplotlib/seaborn for visualization. A fixed random seed (42) ensures reproducibility of results across runs.

In [ ]:
# Basics
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_curve, auc, roc_auc_score)
import matplotlib.pyplot as plt
import seaborn as sns

# Stable random seed
RANDOM_STATE = 42
sns.set_style('whitegrid')

## 2. Data Loading and Preprocessing

### 2.1 Data loading and quick inspection

The Breast Cancer Wisconsin dataset contains 30 numerical features computed from digitized images of breast mass, with 569 samples classified as malignant (0) or benign (1). This cell loads the dataset and displays basic information about its structure and class distribution.

In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target
feature_names = X.columns.tolist()

print('Feature shape:', X.shape)
print('Target distribution:')
print(y.value_counts(normalize=True).rename('proportion'))

# quick head
X.head()

### 2.2 Preprocessing

Feature scaling is crucial for many machine learning algorithms. StandardScaler transforms features to have zero mean and unit variance, ensuring all features contribute equally to distance-based algorithms (like SVM) and gradient-based optimization (like Logistic Regression and Neural Networks).

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=feature_names)
X_scaled.head()

## 3. Exploratory Data Analysis (EDA)

Exploratory analysis helps us understand the data structure, feature distributions, class separability, and potential relationships. This visualization suite includes scatter plots to assess class separability, distribution plots to check for class imbalance, and correlation analysis to identify the most informative features.

In [ ]:
# Enhanced EDA with multiple visualizations
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Plot 1: mean radius vs mean texture by class
scatter = axes[0, 0].scatter(X['mean radius'], X['mean texture'], c=y, s=20, cmap='coolwarm', alpha=0.6)
axes[0, 0].set_xlabel('Mean Radius')
axes[0, 0].set_ylabel('Mean Texture')
axes[0, 0].set_title('Mean Radius vs Mean Texture (by class)')
plt.colorbar(scatter, ax=axes[0, 0])

# Plot 2: mean area vs mean perimeter by class
scatter2 = axes[0, 1].scatter(X['mean area'], X['mean perimeter'], c=y, s=20, cmap='viridis', alpha=0.6)
axes[0, 1].set_xlabel('Mean Area')
axes[0, 1].set_ylabel('Mean Perimeter')
axes[0, 1].set_title('Mean Area vs Mean Perimeter (by class)')
plt.colorbar(scatter2, ax=axes[0, 1])

# Plot 3: Distribution of a key feature by class
axes[0, 2].hist([X[y==0]['mean radius'], X[y==1]['mean radius']], 
                label=['Malignant', 'Benign'], bins=20, alpha=0.7)
axes[0, 2].set_xlabel('Mean Radius')
axes[0, 2].set_ylabel('Frequency')
axes[0, 2].set_title('Mean Radius Distribution by Class')
axes[0, 2].legend()

# Plot 4: Class distribution
class_counts = y.value_counts()
axes[1, 0].bar(['Malignant (0)', 'Benign (1)'], class_counts, color=['coral', 'skyblue'])
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('Class Distribution')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Plot 5: Boxplots for selected features
selected_features = ['mean radius', 'mean texture', 'mean area', 'mean perimeter']
box_data = [X_scaled[feature] for feature in selected_features]
axes[1, 1].boxplot(box_data, labels=selected_features)
axes[1, 1].set_ylabel('Standardjized Values')
axes[1, 1].set_title('Distribution of Key Features')
axes[1, 1].tick_params(axis='x', rotation=45)

# Plot 6: Feature correlations with target
correlations_with_target = X.corrwith(pd.Series(y, index=X.index)).sort_values(ascending=False)[:10]
axes[1, 2].barh(range(len(correlations_with_target)), correlations_with_target.values)
axes[1, 2].set_yticks(range(len(correlations_with_target)))
axes[1, 2].set_yticklabels(correlations_with_target.index, fontsize=8)
axes[1, 2].set_xlabel('Correlation with Target')
axes[1, 2].set_title('Top 10 Features Correlated with Target')
axes[1, 2].invert_yaxis()
axes[1, 2].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print('\nTop features correlated with target:')
print(correlations_with_target.head(10))

### 3.1 Correlation matrix analysis

Understanding feature correlations is essential for model interpretation and feature engineering. The heatmaps reveal which features are highly correlated with each other (potential redundancy) and which are most strongly associated with the target variable (potential predictive power).

In [ ]:
# Enhanced correlation matrix visualization
corr = X.corr()

# Select top correlated features for better visualization
top_features = correlations_with_target.head(15).index
corr_subset = X[top_features].corr()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Full correlation heatmap
im1 = axes[0].imshow(corr, cmap='coolwarm', aspect='auto', vmin=-1, vmax=1)
axes[0].set_xticks(range(0, len(feature_names), 5))
axes[0].set_yticks(range(0, len(feature_names), 5))
axes[0].set_xticklabels([feature_names[i] for i in range(0, len(feature_names), 5)], 
                        rotation=90, fontsize=6)
axes[0].set_yticklabels([feature_names[i] for i in range(0, len(feature_names), 5)], fontsize=6)
axes[0].set_title('Full Feature Correlation Matrix')
plt.colorbar(im1, ax=axes[0])

# Top features correlation heatmap with annotations
sns.heatmap(corr_subset, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, ax=axes[1], cbar_kws={'shrink': 0.8})
axes[1].set_title('Top 15 Features Correlation Matrix')
axes[1].tick_params(axis='x', rotation=45)
axes[1].tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.show()

## 4. Train/Test Split and Model Evaluation Framework

We split the data into training (80%) and testing (20%) sets with stratification to maintain class proportions. The evaluation framework includes a helper function that trains models, performs hyperparameter tuning via GridSearchCV, and computes comprehensive metrics for fair comparison across different algorithms.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)

### 4.1 Model training and evaluation function

This utility function standardizes the training and evaluation process across all models. It performs GridSearchCV for hyperparameter optimization, computes cross-validation scores to assess model stability, calculates multiple metrics (accuracy, precision, recall, F1, ROC-AUC), and stores predictions for detailed analysis.

In [ ]:
def run_classifier(name, model, X_train, y_train, X_test, y_test, do_grid=False, param_grid=None):
    """Run training, prediction and evaluation for a classifier.
    If do_grid=True and param_grid provided, perform GridSearchCV (with 5-fold CV).
    Returns a dictionary with metrics and the fitted estimator."""
    if do_grid and (param_grid is not None):
        gs = GridSearchCV(model, param_grid, cv=5, n_jobs=-1, scoring='accuracy', verbose=1)
        gs.fit(X_train, y_train)
        best = gs.best_estimator_
        fitted = best
        extra = {'best_params': gs.best_params_, 'cv_best_score': gs.best_score_}
        print(f"\n{name} - Best parameters: {gs.best_params_}")
        print(f"{name} - Best CV accuracy: {gs.best_score_:.4f}")
    else:
        fitted = model.fit(X_train, y_train)
        extra = {}
    
    # Cross-validation scores
    cv_scores = cross_val_score(fitted, X_train, y_train, cv=5, scoring='accuracy')
    extra['cv_scores'] = cv_scores
    extra['cv_mean'] = cv_scores.mean()
    extra['cv_std'] = cv_scores.std()
    
    preds = fitted.predict(X_test)
    
    # Get prediction probabilities for ROC curve (if available)
    if hasattr(fitted, 'predict_proba'):
        y_proba = fitted.predict_proba(X_test)[:, 1]
        extra['y_proba'] = y_proba
        extra['roc_auc'] = roc_auc_score(y_test, y_proba)
    elif hasattr(fitted, 'decision_function'):
        y_scores = fitted.decision_function(X_test)
        extra['y_scores'] = y_scores
        extra['roc_auc'] = roc_auc_score(y_test, y_scores)
    
    metrics = {
        'accuracy': accuracy_score(y_test, preds),
        'precision_macro': precision_score(y_test, preds, average='macro', zero_division=0),
        'precision_binary': precision_score(y_test, preds, average='binary', zero_division=0),
        'recall_macro': recall_score(y_test, preds, average='macro', zero_division=0),
        'recall_binary': recall_score(y_test, preds, average='binary', zero_division=0),
        'f1_macro': f1_score(y_test, preds, average='macro', zero_division=0),
        'f1_binary': f1_score(y_test, preds, average='binary', zero_division=0),
        'confusion_matrix': confusion_matrix(y_test, preds),
        'classification_report': classification_report(y_test, preds, zero_division=0),
        'predictions': preds
    }
    res = {'name': name, 'estimator': fitted, 'metrics': metrics}
    res.update(extra)
    return res

## 5. Model Training and Comparison

We evaluate six different classification algorithms, each with different learning paradigms and complexity levels:
- **Logistic Regression**: Linear model
- **Decision Tree**: Non-linear, rule-based, prone to overfitting
- **Random Forest**: Ensemble of trees, robust and powerful
- **SVM**: Kernel-based, effective in high-dimensional spaces
- **Gradient Boosting**: Sequential ensemble, highly accurate
- **MLP (Neural Network)**: Flexible, can model complex patterns

Each model undergoes hyperparameter tuning to find optimal configurations.

### 5.0 Model configuration and training

For each model, we define a comprehensive hyperparameter grid. GridSearchCV performs 5-fold cross-validation on all parameter combinations to find the best configuration. This systematic approach ensures we're comparing models at their optimal performance rather than with default settings.

In [ ]:
# Enhanced model configuration with hyperparameter tuning
models_config = {
    'Logistic Regression': {
        'model': LogisticRegression(max_iter=15000, random_state=RANDOM_STATE),
        'grid': {
            'C': [0.01, 0.1, 1.0, 10.0, 100.0],
            'penalty': ['l1', 'l2'],
            'solver': ['liblinear', 'saga']
        }
    },
    'Decision Tree': {
        'model': DecisionTreeClassifier(random_state=RANDOM_STATE),
        'grid': {
            'max_depth': [3, 5, 7, 10, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'criterion': ['gini', 'entropy']
        }
    },
    'Random Forest': {
        'model': RandomForestClassifier(random_state=RANDOM_STATE),
        'grid': {
            'n_estimators': [50, 100, 200],
            'max_depth': [5, 10, None],
            'min_samples_split': [2, 5],
            'criterion': ['gini', 'entropy']
        }
    },
    'SVM': {
        'model': SVC(random_state=RANDOM_STATE, probability=True),
        'grid': {
            'C': [0.01, 0.05, 0.1, 1, 10, 100],
            'gamma': ['scale', 'auto', 0.001, 0.01],
            'kernel': ['rbf', 'linear', 'poly', 'sigmoid']
        }
    },
    'Gradient Boosting': {
        'model': GradientBoostingClassifier(
            random_state=RANDOM_STATE,
            n_iter_no_change=10,
            validation_fraction=0.1
            ),
        'grid': {
            'n_estimators': [50, 100, 200],
            'learning_rate': [0.01, 0.1, 0.2],
            'max_depth': [3, 5, 7]
        }
    },
    'MLP': {
        'model': MLPClassifier(
            max_iter=15000, 
            random_state=RANDOM_STATE,
            early_stopping=True,
            n_iter_no_change=10,
            validation_fraction=0.1
            ),
        'grid': {
            'hidden_layer_sizes': [(50, 50, 15), (15, 50, 15),(50, 50), (100, 50)],
            'alpha': [0.0005, 0.001, 0.01],
            'learning_rate': ['constant', 'adaptive'],
            'activation': ['relu', 'tanh', 'logistic']
        }
    }
}

results = {}
for name, config in models_config.items():
    print(f"\n{'='*60}")
    print(f"Training {name}...")
    print('='*60)
    res = run_classifier(
        name, 
        config['model'], 
        X_train, y_train, 
        X_test, y_test,
        do_grid=True,
        param_grid=config['grid']
    )
    results[name] = res
    print(f"{name}: accuracy={res['metrics']['accuracy']:.4f}, f1={res['metrics']['f1_binary']:.4f}")
    if 'cv_mean' in res:
        print(f"{name}: CV accuracy={res['cv_mean']:.4f} (+/- {res['cv_std']:.4f})")
    if 'roc_auc' in res:
        print(f"{name}: ROC AUC={res['roc_auc']:.4f}")

### 5.1 Model Performance Comparison

This section aggregates and visualizes performance metrics across all models. By comparing accuracy, precision, recall, F1-score, and ROC-AUC side-by-side, we can identify which models excel at different aspects of classification and understand the trade-offs between metrics.

In [ ]:
# Create comprehensive comparison dataframe
comparison_data = []
for name, res in results.items():
    row = {
        'Model': name,
        'Accuracy': res['metrics']['accuracy'],
        'Precision': res['metrics']['precision_binary'],
        'Recall': res['metrics']['recall_binary'],
        'F1': res['metrics']['f1_binary'],
    }
    if 'cv_mean' in res:
        row['CV_Accuracy_Mean'] = res['cv_mean']
        row['CV_Accuracy_Std'] = res['cv_std']
    if 'roc_auc' in res:
        row['ROC_AUC'] = res['roc_auc']
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('Accuracy', ascending=False)
print('\nModel Performance Comparison:')
print('='*100)
print(comparison_df.to_string(index=False))
print('='*100)

# Visualize metrics comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Accuracy comparison
axes[0, 0].barh(comparison_df['Model'], comparison_df['Accuracy'], color='skyblue')
axes[0, 0].set_xlabel('Accuracy')
axes[0, 0].set_title('Accuracy Comparison')
axes[0, 0].set_xlim([0.9, 1.0])
axes[0, 0].grid(True, alpha=0.3, axis='x')

# Precision comparison
axes[0, 1].barh(comparison_df['Model'], comparison_df['Precision'], color='coral')
axes[0, 1].set_xlabel('Precision')
axes[0, 1].set_title('Precision Comparison')
axes[0, 1].set_xlim([0.9, 1.0])
axes[0, 1].grid(True, alpha=0.3, axis='x')

# Recall comparison
axes[0, 2].barh(comparison_df['Model'], comparison_df['Recall'], color='lightgreen')
axes[0, 2].set_xlabel('Recall')
axes[0, 2].set_title('Recall Comparison')
axes[0, 2].set_xlim([0.9, 1.0])
axes[0, 2].grid(True, alpha=0.3, axis='x')

# F1 Score comparison
axes[1, 0].barh(comparison_df['Model'], comparison_df['F1'], color='plum')
axes[1, 0].set_xlabel('F1 Score')
axes[1, 0].set_title('F1 Score Comparison')
axes[1, 0].set_xlim([0.9, 1.0])
axes[1, 0].grid(True, alpha=0.3, axis='x')

# ROC AUC comparison
if 'ROC_AUC' in comparison_df.columns:
    roc_data = comparison_df.dropna(subset=['ROC_AUC'])
    axes[1, 1].barh(roc_data['Model'], roc_data['ROC_AUC'], color='gold')
    axes[1, 1].set_xlabel('ROC AUC')
    axes[1, 1].set_title('ROC AUC Comparison')
    axes[1, 1].set_xlim([0.9, 1.0])
    axes[1, 1].grid(True, alpha=0.3, axis='x')

# All metrics together
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1']
x = np.arange(len(comparison_df))
width = 0.2
for i, metric in enumerate(metrics_to_plot):
    axes[1, 2].bar(x + i*width, comparison_df[metric], width, label=metric, alpha=0.8)
axes[1, 2].set_xlabel('Model')
axes[1, 2].set_ylabel('Score')
axes[1, 2].set_title('All Metrics Comparison')
axes[1, 2].set_xticks(x + width * 1.5)
axes[1, 2].set_xticklabels(comparison_df['Model'], rotation=45, ha='right')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

### 5.2 Confusion Matrices and Classification Reports

Confusion matrices show the detailed breakdown of correct and incorrect predictions for each class. They reveal whether models make specific types of errors (false positives vs. false negatives), which is critical in medical diagnosis where different error types have different costs.

In [ ]:
# Enhanced confusion matrices visualization
n_models = len(results)
n_cols = 3
n_rows = (n_models + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
axes = axes.flatten() if n_models > 1 else [axes]

for idx, (name, info) in enumerate(results.items()):
    cm = info['metrics']['confusion_matrix']
    
    # Create heatmap
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], 
                cbar_kws={'shrink': 0.8}, square=True)
    axes[idx].set_ylabel('True Label')
    axes[idx].set_xlabel('Predicted Label')
    axes[idx].set_title(f'{name}\nAccuracy: {info["metrics"]["accuracy"]:.4f}')
    axes[idx].set_yticklabels(['Malignant', 'Benign'], rotation=0)
    axes[idx].set_xticklabels(['Malignant', 'Benign'], rotation=0)

# Hide extra subplots
for idx in range(n_models, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

# Print detailed classification reports
for name, info in results.items():
    print('\n' + '='*60)
    print(f'{name} - Classification Report')
    print('='*60)
    print(info['metrics']['classification_report'])

### 5.3 ROC Curves Analysis

Receiver Operating Characteristic (ROC) curves visualize the trade-off between true positive rate and false positive rate across different classification thresholds. The Area Under the Curve (AUC) provides a single metric to compare model discrimination ability, with values closer to 1.0 indicating better performance.

In [ ]:
# ROC curves for all models
plt.figure(figsize=(10, 8))

for name, res in results.items():
    if 'y_proba' in res:
        fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {roc_auc:.3f})')
    elif 'y_scores' in res:
        fpr, tpr, _ = roc_curve(y_test, res['y_scores'])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier (AUC = 0.5)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves Comparison', fontsize=14)
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print AUC scores
print('\nROC AUC Scores:')
print('='*40)
for name, res in results.items():
    if 'roc_auc' in res:
        print(f'{name}: {res["roc_auc"]:.4f}')

### 5.4 Cross-Validation Performance Analysis

Cross-validation assesses how well models generalize to unseen data by training and testing on different data subsets. Comparing CV scores with test accuracy reveals whether models are overfitting (high training score, low test score) or generalizing well (similar scores).

In [ ]:
# Cross-validation comparison for all models
cv_data = []
for name, res in results.items():
    if 'cv_scores' in res:
        cv_data.append({
            'Model': name,
            'CV_Mean': res['cv_mean'],
            'CV_Std': res['cv_std'],
            'Test_Accuracy': res['metrics']['accuracy']
        })

cv_df = pd.DataFrame(cv_data).sort_values('CV_Mean', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Box plot of cross-validation scores
cv_scores_list = [results[name]['cv_scores'] for name in cv_df['Model']]
bp = axes[0].boxplot(cv_scores_list, labels=cv_df['Model'], patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('lightblue')
axes[0].set_ylabel('Accuracy Score')
axes[0].set_title('Cross-Validation Score Distribution')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].set_ylim([0.9, 1.0])

# CV vs Test performance
x_pos = np.arange(len(cv_df))
width = 0.35
axes[1].bar(x_pos - width/2, cv_df['CV_Mean'], width, label='CV Mean Accuracy', 
            alpha=0.8, yerr=cv_df['CV_Std'], capsize=5)
axes[1].bar(x_pos + width/2, cv_df['Test_Accuracy'], width, label='Test Accuracy', alpha=0.8)
axes[1].set_xlabel('Model')
axes[1].set_ylabel('Accuracy Score')
axes[1].set_title('Cross-Validation vs Test Performance')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(cv_df['Model'], rotation=45, ha='right')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_ylim([0.9, 1.0])

plt.tight_layout()
plt.show()

print('\nCross-Validation Summary:')
print(cv_df.to_string(index=False))

### 5.5 Feature Importance Analysis

Understanding which features drive predictions is crucial for model interpretation. Tree-based models provide feature importance scores, while logistic regression provides coefficients. This analysis reveals which tumor characteristics are most predictive of malignancy.

In [ ]:
# Analyze feature importance for tree-based and logistic models
tree_models = ['Decision Tree', 'Random Forest', 'Gradient Boosting']
available_tree_models = [name for name in tree_models if name in results]

if available_tree_models:
    n_models = len(available_tree_models)
    fig, axes = plt.subplots(1, n_models, figsize=(7*n_models, 6))
    if n_models == 1:
        axes = [axes]
    
    for idx, name in enumerate(available_tree_models):
        estimator = results[name]['estimator']
        # Get feature importances
        if hasattr(estimator, 'feature_importances_'):
            importances = estimator.feature_importances_
            indices = np.argsort(importances)[::-1][:15]  # Top 15 features
            
            axes[idx].barh(range(len(indices)), importances[indices], align='center')
            axes[idx].set_yticks(range(len(indices)))
            axes[idx].set_yticklabels([feature_names[i] for i in indices], fontsize=9)
            axes[idx].set_xlabel('Feature Importance', fontsize=10)
            axes[idx].set_title(f'{name}\nTop 15 Features', fontsize=11)
            axes[idx].invert_yaxis()
            axes[idx].grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()
    
    # Print feature importance rankings
    for name in available_tree_models:
        estimator = results[name]['estimator']
        if hasattr(estimator, 'feature_importances_'):
            importances = estimator.feature_importances_
            indices = np.argsort(importances)[::-1][:10]
            print(f"\n{name} - Top 10 Feature Importance:")
            for i, idx in enumerate(indices):
                print(f"  {i+1}. {feature_names[idx]}: {importances[idx]:.4f}")

# Logistic Regression Coefficients
if 'Logistic Regression' in results:
    lr_model = results['Logistic Regression']['estimator']
    if hasattr(lr_model, 'coef_'):
        coef = lr_model.coef_[0]
        coef_abs = np.abs(coef)
        indices = np.argsort(coef_abs)[::-1][:15]
        
        plt.figure(figsize=(10, 6))
        colors = ['green' if coef[i] > 0 else 'red' for i in indices]
        plt.barh(range(len(indices)), coef[indices], align='center', color=colors, alpha=0.7)
        plt.yticks(range(len(indices)), [feature_names[i] for i in indices], fontsize=9)
        plt.xlabel('Coefficient Value', fontsize=10)
        plt.title('Logistic Regression - Top 15 Feature Coefficients\n(Green=Positive, Red=Negative)', fontsize=11)
        plt.axvline(x=0, color='black', linestyle='--', linewidth=1)
        plt.grid(True, alpha=0.3, axis='x')
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.show()
        
        print(f"\nLogistic Regression - Top 10 Features by Absolute Coefficient:")
        for i, idx in enumerate(indices[:10]):
            print(f"  {i+1}. {feature_names[idx]}: {coef[idx]:.4f}")

### 5.6 Learning Curves Analysis

Learning curves show how model performance changes with training set size. They help diagnose whether models would benefit from more data (high bias/underfitting) or are overfitting (large gap between training and validation curves). Convergence of both curves indicates good generalization.

In [ ]:
# Learning curves for all models
# Sort models by accuracy score
all_models = sorted(results.keys(), key=lambda n: results[n]['metrics']['accuracy'], reverse=True)

# Create grid layout for all models
n_models = len(all_models)
n_cols = 3
n_rows = (n_models + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5*n_rows))
axes = axes.flatten() if n_models > 1 else [axes]

for idx, name in enumerate(all_models):
    model = results[name]['estimator']
    
    train_sizes, train_scores, val_scores = learning_curve(
        model, X_train, y_train, cv=5, 
        train_sizes=np.linspace(0.1, 1.0, 10),
        scoring='accuracy',
        n_jobs=-1
    )
    
    train_mean = np.mean(train_scores, axis=1)
    train_std = np.std(train_scores, axis=1)
    val_mean = np.mean(val_scores, axis=1)
    val_std = np.std(val_scores, axis=1)
    
    axes[idx].plot(train_sizes, train_mean, 'o-', label='Training score', linewidth=2, color='blue')
    axes[idx].fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='blue')
    axes[idx].plot(train_sizes, val_mean, 'o-', label='Cross-validation score', linewidth=2, color='orange')
    axes[idx].fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color='orange')
    
    axes[idx].set_xlabel('Training Set Size', fontsize=10)
    axes[idx].set_ylabel('Accuracy Score', fontsize=10)
    axes[idx].set_title(f'Learning Curve - {name}\nAccuracy={results[name]["metrics"]["accuracy"]:.4f}', fontsize=11)
    axes[idx].legend(loc='lower right', fontsize=9)
    axes[idx].grid(True, alpha=0.3)
    axes[idx].set_ylim([0.85, 1.05])  # Set reasonable y-axis limits

# Hide extra subplots
for idx in range(n_models, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

print(f"\nLearning curves generated for all {n_models} models.")
print("Models ordered by accuracy score (best to worst).")

### 5.6.1 Cross-Validation Stability Analysis

This detailed analysis performs 10-fold cross-validation to assess model consistency across different data splits. Models with low variance (small standard deviation) are more reliable and stable, while high variance indicates sensitivity to training data composition.

In [ ]:
# Analyze cross-validation stability with multiple visualizations

# 1. Detailed CV scores analysis with MORE FOLDS for better stability
from sklearn.model_selection import cross_validate

print("="*80)
print("CROSS-VALIDATION STABILITY ANALYSIS")
print("="*80)

cv_detailed_results = {}

for name in results.keys():
    model = results[name]['estimator']
    
    # Perform cross-validation with 10 folds (more folds = more stable estimates)
    cv_results = cross_validate(
        model, X_train, y_train,
        cv=10,  # Increased from 5 to 10 folds
        scoring=['accuracy', 'precision', 'recall', 'f1'],
        return_train_score=True,
        n_jobs=-1
    )
    
    cv_detailed_results[name] = {
        'test_accuracy': cv_results['test_accuracy'],
        'test_precision': cv_results['test_precision'],
        'test_recall': cv_results['test_recall'],
        'test_f1': cv_results['test_f1'],
        'train_accuracy': cv_results['train_accuracy']
    }
    
    print(f"\n{name}:")
    print(f"  Test Accuracy:  {cv_results['test_accuracy'].mean():.4f} ± {cv_results['test_accuracy'].std():.4f}")
    print(f"  Test Precision: {cv_results['test_precision'].mean():.4f} ± {cv_results['test_precision'].std():.4f}")
    print(f"  Test Recall:    {cv_results['test_recall'].mean():.4f} ± {cv_results['test_recall'].std():.4f}")
    print(f"  Test F1:        {cv_results['test_f1'].mean():.4f} ± {cv_results['test_f1'].std():.4f}")
    print(f"  Train Accuracy: {cv_results['train_accuracy'].mean():.4f} ± {cv_results['train_accuracy'].std():.4f}")
    
    # Calculate overfitting metric
    overfit = cv_results['train_accuracy'].mean() - cv_results['test_accuracy'].mean()
    print(f"  Overfitting (Train-Test): {overfit:.4f}")

print("="*80)

### 5.6.2 Comprehensive stability metrics

This section computes detailed cross-validation metrics across multiple performance measures (accuracy, precision, recall, F1) and calculates the gap between training and test scores to quantify overfitting.

In [ ]:
# Visualization 1: Violin plots for CV score distributions
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

metrics_to_plot = ['test_accuracy', 'test_precision', 'test_recall', 'test_f1']
metric_titles = ['Accuracy', 'Precision', 'Recall', 'F1 Score']

for idx, (metric, title) in enumerate(zip(metrics_to_plot, metric_titles)):
    ax = axes[idx // 2, idx % 2]
    
    # Prepare data for violin plot
    data_for_violin = []
    labels = []
    for name in sorted(cv_detailed_results.keys(), 
                      key=lambda n: cv_detailed_results[n][metric].mean(), 
                      reverse=True):
        data_for_violin.append(cv_detailed_results[name][metric])
        labels.append(name)
    
    # Create violin plot
    parts = ax.violinplot(data_for_violin, positions=range(len(labels)), 
                          showmeans=True, showmedians=True, widths=0.7)
    
    # Color the violins
    for pc in parts['bodies']:
        pc.set_facecolor('lightblue')
        pc.set_alpha(0.7)
    
    # Add scatter points for individual CV scores
    for i, scores in enumerate(data_for_violin):
        x = np.random.normal(i, 0.04, size=len(scores))
        ax.scatter(x, scores, alpha=0.5, s=30, color='darkblue')
    
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel(f'{title} Score', fontsize=10)
    ax.set_title(f'10-Fold CV {title} Distribution', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim([0.85, 1.05])

plt.tight_layout()
plt.show()

In [ ]:
# Visualization 2: CV Stability Heatmap
fig, ax = plt.subplots(figsize=(14, 6))

# Create matrix of CV scores (models x folds)
model_names = sorted(cv_detailed_results.keys(), 
                     key=lambda n: cv_detailed_results[n]['test_accuracy'].mean(), 
                     reverse=True)
cv_matrix = np.array([cv_detailed_results[name]['test_accuracy'] for name in model_names])

# Create heatmap
im = ax.imshow(cv_matrix, cmap='RdYlGn', aspect='auto', vmin=0.90, vmax=1.0)

# Set ticks and labels
ax.set_yticks(range(len(model_names)))
ax.set_yticklabels(model_names, fontsize=10)
ax.set_xticks(range(cv_matrix.shape[1]))
ax.set_xticklabels([f'Fold {i+1}' for i in range(cv_matrix.shape[1])], fontsize=9)
ax.set_xlabel('Cross-Validation Fold', fontsize=11)
ax.set_ylabel('Model', fontsize=11)
ax.set_title('Accuracy Across 10-Fold Cross-Validation\n(Darker Green = Better Performance)', 
             fontsize=12, fontweight='bold')

# Add text annotations
for i in range(len(model_names)):
    for j in range(cv_matrix.shape[1]):
        text = ax.text(j, i, f'{cv_matrix[i, j]:.3f}',
                      ha="center", va="center", color="black", fontsize=7)

# Add colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Accuracy Score', fontsize=10)

plt.tight_layout()
plt.show()

### 5.7 Best Model Detailed Analysis

After comparing all models, we identify the best performer (prioritizing recall to minimize false negatives in cancer detection) and provide a comprehensive analysis including confusion matrix, metrics breakdown, ROC curve, and prediction distribution.

In [ ]:
# Find best model by recall, break ties with accuracy
recalls = [results[n]['metrics']['recall_binary'] for n in results]
max_recall = max(recalls)
# Get all models with max recall
candidates = [n for n in results if results[n]['metrics']['recall_binary'] == max_recall]
# If tie, pick highest accuracy among them
best_name = max(candidates, key=lambda n: results[n]['metrics']['accuracy'])
best = results[best_name]

print('='*60)
print(f'Best Model: {best_name}')
print('='*60)
print(f"Accuracy: {best['metrics']['accuracy']:.4f}")
print(f"Precision: {best['metrics']['precision_binary']:.4f}")
print(f"Recall: {best['metrics']['recall_binary']:.4f}")
print(f"F1 Score: {best['metrics']['f1_binary']:.4f}")
if 'roc_auc' in best:
    print(f"ROC AUC: {best['roc_auc']:.4f}")
if 'best_params' in best:
    print(f"\nBest Hyperparameters:")
    for param, value in best['best_params'].items():
        print(f"  {param}: {value}")
if 'cv_mean' in best:
    print(f"\nCross-Validation Accuracy: {best['cv_mean']:.4f} (+/- {best['cv_std']:.4f})")

# Detailed visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Confusion Matrix
cm = best['metrics']['confusion_matrix']
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0], 
            cbar_kws={'shrink': 0.8}, square=True, annot_kws={'size': 14})
axes[0, 0].set_ylabel('True Label', fontsize=11)
axes[0, 0].set_xlabel('Predicted Label', fontsize=11)
axes[0, 0].set_title(f'{best_name} - Confusion Matrix', fontsize=12)
axes[0, 0].set_yticklabels(['Malignant', 'Benign'], rotation=0)
axes[0, 0].set_xticklabels(['Malignant', 'Benign'], rotation=0)

# Metrics bar chart
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
metrics_values = [
    best['metrics']['accuracy'],
    best['metrics']['precision_binary'],
    best['metrics']['recall_binary'],
    best['metrics']['f1_binary']
]
colors_bar = ['skyblue', 'coral', 'lightgreen', 'plum']
axes[0, 1].bar(metrics_names, metrics_values, color=colors_bar, alpha=0.8)
axes[0, 1].set_ylabel('Score', fontsize=11)
axes[0, 1].set_title(f'{best_name} - Performance Metrics', fontsize=12)
axes[0, 1].set_ylim([0.9, 1.0])
axes[0, 1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(metrics_values):
    axes[0, 1].text(i, v + 0.005, f'{v:.4f}', ha='center', fontsize=9)

# ROC Curve
if 'y_proba' in best:
    fpr, tpr, _ = roc_curve(y_test, best['y_proba'])
    roc_auc_val = auc(fpr, tpr)
    axes[1, 0].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc_val:.3f})')
    axes[1, 0].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
    axes[1, 0].set_xlim([0.0, 1.0])
    axes[1, 0].set_ylim([0.0, 1.05])
    axes[1, 0].set_xlabel('False Positive Rate', fontsize=11)
    axes[1, 0].set_ylabel('True Positive Rate', fontsize=11)
    axes[1, 0].set_title(f'{best_name} - ROC Curve', fontsize=12)
    axes[1, 0].legend(loc='lower right', fontsize=9)
    axes[1, 0].grid(True, alpha=0.3)
elif 'y_scores' in best:
    fpr, tpr, _ = roc_curve(y_test, best['y_scores'])
    roc_auc_val = auc(fpr, tpr)
    axes[1, 0].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc_val:.3f})')
    axes[1, 0].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
    axes[1, 0].set_xlim([0.0, 1.0])
    axes[1, 0].set_ylim([0.0, 1.05])
    axes[1, 0].set_xlabel('False Positive Rate', fontsize=11)
    axes[1, 0].set_ylabel('True Positive Rate', fontsize=11)
    axes[1, 0].set_title(f'{best_name} - ROC Curve', fontsize=12)
    axes[1, 0].legend(loc='lower right', fontsize=9)
    axes[1, 0].grid(True, alpha=0.3)

# Prediction distribution
preds = best['metrics']['predictions']
pred_counts = pd.Series(preds).value_counts().sort_index()
true_counts = pd.Series(y_test.values).value_counts().sort_index()
x_pos = np.arange(2)
width = 0.35
axes[1, 1].bar(x_pos - width/2, true_counts, width, label='True Labels', alpha=0.8, color='lightblue')
axes[1, 1].bar(x_pos + width/2, pred_counts, width, label='Predictions', alpha=0.8, color='lightcoral')
axes[1, 1].set_ylabel('Count', fontsize=11)
axes[1, 1].set_title(f'{best_name} - True vs Predicted Distribution', fontsize=12)
axes[1, 1].set_xticks(x_pos)
axes[1, 1].set_xticklabels(['Malignant (0)', 'Benign (1)'])
axes[1, 1].legend(fontsize=9)
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 6. Discussion: Model Performance and Characteristics

### 6.1 Performance Overview

All models achieved strong performance on the Breast Cancer Wisconsin dataset, with test accuracies ranging from 92% to 98%. However, significant differences emerged in terms of complexity, stability, generalization capacity, and the bias-variance trade-off.

**Performance Ranking (by test accuracy):**
1. **SVM**: 98% (best overall)
2. **Logistic Regression**: 97% 
3. **Random Forest**: 96%
4. **Gradient Boosting**: 93%
5. **MLP**: 93%
6. **Decision Tree**: 92%

### 6.2 Model-by-Model Analysis

#### **Support Vector Machine (SVM) - Winner** ✓
- **Test Accuracy**: 98% | **CV Accuracy**: 97.81% ± 2.57%
- **Complexity**: High (kernel-based, RBF kernel selected)
- **Generalization**: Excellent - very low overfitting (0.94%)
- **Stability**: Excellent - low variance across folds (std = 2.57%)
- **Why it performs best**: The RBF kernel effectively captures non-linear decision boundaries in the 30-dimensional feature space. Margin maximization finds optimal separation between classes. Proper C and gamma tuning prevents overfitting while maintaining high accuracy.
- **Key strength**: Perfect balance of high performance and generalization (train-test gap of only 0.94%)
- **Limitations**: Computationally expensive for very large datasets, requires careful hyperparameter tuning

#### **Logistic Regression - Strong Runner-up** ✓
- **Test Accuracy**: 97% | **CV Accuracy**: 98.03% ± 2.47%
- **Complexity**: Low (linear decision boundary)
- **Generalization**: Outstanding - minimal overfitting (0.58%)
- **Stability**: Excellent - very low variance (std = 2.47%)
- **Why it performs so well**: The dataset features are surprisingly linearly separable after standardization. L1/L2 regularization prevents overfitting. Excellent recall (99.66% - tie with SVM) makes it ideal for medical diagnosis.
- **Key strength**: Simplest model with near-best performance, highly interpretable, fastest training
- **Notable**: Actually has slightly HIGHER CV accuracy (98.03%) than test (97%), showing excellent generalization

#### **Random Forest - Consistent Performer**
- **Test Accuracy**: 96% | **CV Accuracy**: 96.05% ± 3.36%
- **Complexity**: High (ensemble of 200 trees)
- **Generalization**: Good but shows moderate overfitting (3.76%)
- **Stability**: Good but higher variance (std = 3.36%)
- **Why it performs well**: Bootstrap aggregation and feature randomness reduce individual tree variance. Robust to outliers and noise.
- **Trade-off**: More complex than needed for this dataset - the overfitting gap suggests the ensemble captures some noise
- **Limitations**: Moderate overfitting compared to simpler models

#### **MLP (Neural Network) - Surprisingly Balanced**
- **Test Accuracy**: 93% | **CV Accuracy**: 96.27% ± 2.60%
- **Complexity**: Very high (multi-layer architecture)
- **Generalization**: Exceptional - actually shows slight underfitting
- **Stability**: Good - moderate variance (std = 2.60%)
- **Why it performs well**: Multiple hidden layers learn hierarchical feature representations. Early stopping and dropout regularization prevent overfitting effectively.
- **Limitation**: High computacional cost.

#### **Gradient Boosting - Moderate Performance**
- **Test Accuracy**: 93% | **CV Accuracy**: 95.16% ± 2.93%
- **Complexity**: Very high (sequential ensemble)
- **Generalization**: Moderate - shows overfitting (4.20%)
- **Stability**: Good - moderate variance (std = 2.93%)
- **Why it underperforms expectations**: Sequential boosting may be over-focusing on difficult examples. The 4.20% overfitting gap suggests the model is memorizing training patterns rather than generalizing.
- **Reason for underperformance**: The dataset may be too simple for boosting's complexity, or hyperparameters need further tuning

#### **Decision Tree - Weakest Performer**
- **Test Accuracy**: 92% | **CV Accuracy**: 92.98% ± 2.53%
- **Complexity**: Medium (single tree with depth constraints)
- **Generalization**: Poor - significant overfitting (6.10%)
- **Stability**: okay variance but lowest accuracy
- **Why it underperforms**: Single trees are high-variance estimators prone to overfitting. The 6.10% overfitting gap is the highest among all models, indicating it's memorizing training data.
- **Limitations**: Cannot compete with ensemble methods or well-tuned parametric models on this dataset

### 6.3 Key Insights from Results

#### **The Linear Separability Discovery**
The outstanding performance of Logistic Regression (97% accuracy, 98% CV accuracy, only 0.58% overfitting) reveals that this dataset is **remarkably linearly separable**. This is surprising for a medical diagnosis problem and suggests excellent feature engineering in the original dataset construction.

#### **Complexity ≠ Performance**
Contrary to expectations:
- **Simple models won**: SVM (98%) and Logistic Regression (97%) outperformed complex ensembles
- **Gradient Boosting struggled**: Only 93% accuracy with 4.20% overfitting
- **Decision Tree worst**: 92% accuracy with 6.10% overfitting

This demonstrates that **model complexity should match problem complexity**. Over-engineering with complex models when simpler ones suffice leads to overfitting.

#### **Overfitting Analysis**
Models ranked by overfitting (lower is better):
1. **MLP**: -0.06% (slight underfitting!)
2. **Logistic Regression**: 0.58%
3. **SVM**: 0.94%
4. **Random Forest**: 3.76%
5. **Gradient Boosting**: 4.20%
6. **Decision Tree**: 6.10%

#### **Stability Analysis (CV Standard Deviation)**
Models ranked by stability (lower variance is better):
1. **Logistic Regression**: 2.47%
2. **Decision Tree**: 2.53%
3. **SVM**: 2.57%
4. **MLP**: 2.60%
5. **Gradient Boosting**: 2.93%
6. **Random Forest**: 3.36%

All models show good stability (all under 3.4%), but simpler models are slightly more consistent.

### 6.4 Practical Recommendations

#### **For Production Deployment:**
1. **Best choice: SVM** - Highest accuracy (98%), excellent generalization, stable
2. **Alternative: Logistic Regression** - Nearly as good (97%), much faster, more interpretable
3. **Avoid: Decision Tree** - Weakest performance and highest overfitting

#### **For Medical Diagnosis:**
- **Recommended: Logistic Regression** - Exceptional recall (99.66% in CV), minimizes false negatives (critical for cancer detection), interpretable coefficients for clinical trust
- **Alternative: SVM** - Highest overall accuracy if interpretability is not the priority

#### **Model Selection Criteria:**
- **Need interpretability?** → Logistic Regression
- **Need maximum accuracy?** → SVM
- **Limited computational resources?** → Logistic Regression

### 6.5 Conclusion

This analysis revealed a counter-intuitive but important lesson: **simpler models can outperform complex ones when the problem structure is relatively simple**. The Breast Cancer Wisconsin dataset, despite being a medical diagnosis problem, exhibits strong linear separability that favors well-regularized simple models.